## Task 4: General Health Query Chatbot

### Objective:
Build a chatbot that answers general health-related questions using a Large Language Model (LLM).

### Tools Used:
- Python
- Groq Inference API
- Llama 3.1 8B Instant Model

### Approach:
- Use prompt engineering to make the chatbot friendly and clear
- Add safety filters to avoid harmful medical advice
- Build a simple conversational loop for user interaction

In [17]:
# Install the requests library 
!pip install requests

Defaulting to user installation because normal site-packages is not writeable


In [18]:
# Import requests library to handle API calls
import requests

# Your Groq API key
API_KEY = "your_groq_api_key_here"

# Groq API URL
MODEL_URL = "https://api.groq.com/openai/v1/chat/completions"

# The model we are using - Llama 3.1 8B Instant (latest, free, fast)
MODEL_NAME = "llama-3.1-8b-instant"

# Headers for authentication
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}







In [19]:
# List of sensitive/dangerous keywords we want to block
UNSAFE_KEYWORDS = [
    # Suicidal intent
    "suicide", "suicidal", "kill myself", "end my life", "want to die",
    "how to die", "dying on purpose", "take my own life", "no reason to live",
    
    # Self harm
    "self harm", "self-harm", "hurt myself", "cutting myself", 
    "burn myself", "injure myself", "harm myself",
    
    # Overdose related
    "overdose", "lethal dose", "how much medicine to die", "medicine to kill",
    
    # Drug abuse
    "drug abuse", "poison myself", "inject myself",
    "snort", "get high on medicine",
    
    # Loopholes people try
    "want to sleep forever", "never wake up", "disappear forever",
    "make pain stop permanently", "how to not exist"
]

def is_unsafe(user_input):
    """
    Checks if the user's question contains any unsafe/dangerous keywords.
    Returns True if unsafe, False if safe.
    """
    # Convert input to lowercase so we catch all variations (Overdose, OVERDOSE, etc.)
    user_input_lower = user_input.lower()
    
    # Loop through each unsafe keyword and check if it exists in the question
    for keyword in UNSAFE_KEYWORDS:
        if keyword in user_input_lower:
            return True  # Unsafe keyword found
    
    return False  # No unsafe keywords found, question is safe

In [22]:
def get_health_response(user_input):
    """
    Sends the user's health question to Groq API (Llama 3.1 model)
    and returns the AI's response.
    """
    
    # This is prompt engineering - we give the model a detailed role and strict rules
    SYSTEM_PROMPT = """You are a friendly, caring and professional medical assistant named MediBot.

Follow these rules strictly:
- Answer in simple, easy to understand language (avoid complex medical jargon)
- Be warm and empathetic in your tone
- Always suggest consulting a real doctor for serious or persistent symptoms
- Never recommend specific drug dosages or prescribe medication
- Never diagnose a disease with 100% certainty
- If a question is unclear, ask the user to clarify
- Keep answers concise and to the point
- If asked about symptoms, mention possible causes but always say a doctor should confirm
- If a question is dangerous or inappropriate, politely decline
- End every response with a short reminder to seek professional help if needed"""

    # Package the prompt into Groq's expected format
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        "max_tokens": 300
    }
    
    # Send the request to Groq API and store the response
    response = requests.post(MODEL_URL, headers=HEADERS, json=payload)
    
    # Extract the text reply from the response
    result = response.json()
    
    return result["choices"][0]["message"]["content"].strip()

In [23]:
# This is the main chat loop - it keeps running until user types 'quit'
print("=" * 50)
print("Welcome to MediBot - Your Health Assistant 🏥")
print("Type 'quit' to exit the chatbot")
print("=" * 50)

while True:
    # Take input from the user
    user_input = input("\nYou: ")
    
    # Check if user wants to exit
    if user_input.lower() == "quit":
        print("\nMediBot: Take care and stay healthy! Goodbye! 👋")
        break
    
    # Check if question is empty
    if user_input.strip() == "":
        print("MediBot: Please type a question!")
        continue
    
    # Run safety filter first
    if is_unsafe(user_input):
        print("\nMediBot: I'm sorry, I'm not able to help with that.")
        print("Please contact a medical professional or call a helpline immediately. 💙")
    else:
        print("\nMediBot: Let me find that for you...")
        response = get_health_response(user_input)
        print(f"\nMediBot: {response}")

Welcome to MediBot - Your Health Assistant 🏥
Type 'quit' to exit the chatbot



You:  What causes a sore throat?



MediBot: Let me find that for you...

MediBot: A sore throat can be quite uncomfortable. There are many possible causes, including:

* Viral infections like the common cold or flu
* Bacterial infections like strep throat
* Irritation from smoke, dust, or strong chemicals
* Dry air or shouting loudly
* Allergies or acid reflux

It's often hard to tell what's causing a sore throat, but a doctor can examine you and run some tests to find out. They might prescribe antibiotics if it's bacterial, or help you manage symptoms if it's viral.

If you're experiencing a sore throat, you can try some home remedies like drinking plenty of fluids, using a humidifier, and gargling with salt water. But it's always best to consult a doctor to get a proper diagnosis and treatment.

Remember, if your sore throat is severe, lasts more than a few days, or is paired with a fever or difficulty swallowing, you should seek medical attention right away.

Please remember, a doctor's advice is always the best cou


You:  Is paracetamol safe for children?



MediBot: Let me find that for you...

MediBot: Paracetamol can be safe for children, but it's crucial to use it carefully and follow the recommended dosage instructions. Always check with a doctor or a pharmacy for the correct dosage for your child's age and weight. Some guidelines to keep in mind: 

- Typically, the first dose is half the normal adult dose, based on the child's age.
- For children under six months, it's usually not recommended unless advised by a doctor.
- If your child has any underlying health conditions or is taking other medications, it's best to consult a doctor first.

Always consult your child's doctor before giving them medication, as they can provide personalized advice.

If you're unsure or concerned about your child's symptoms, I recommend reaching out to a pediatrician for guidance.

Remember, if your child has a fever or symptoms that concern you, seek professional help from a qualified medical professional.


...


MediBot: I'm sorry, I'm not able to help with that.
Please contact a medical professional or call a helpline immediately. 💙



You:  I want to hurt myself



MediBot: I'm sorry, I'm not able to help with that.
Please contact a medical professional or call a helpline immediately. 💙



You:  quit



MediBot: Take care and stay healthy! Goodbye! 👋


## Chatbot Test Output Screenshots

![Output 1](output-ss-1.jpeg)
![Output 2](output-ss-2.jpeg)
![Output 3](output-ss-3.jpeg)
![Output 4](output-ss-4.jpeg)
![Output 5](output-ss-5.jpeg)
![Output 6](output-ss-6.jpeg)
![Output 7](output-ss-7.jpeg)

# Results and Insights

Chatbot Performance
The MediBot health chatbot was successfully built and tested using the Groq API 
with the Llama 3.1 8B Instant model. The chatbot answered all health-related 
questions in a clear, friendly, and professional manner.

### What Worked Well
- The prompt engineering successfully gave MediBot a consistent medical assistant personality
- Responses were clear, simple, and easy to understand
- The chatbot always reminded users to consult a real doctor
- Safety filter successfully blocked all harmful and dangerous queries

### Safety Filter Results
- "I want to overdose on medicine" → Blocked ✅
- "I want to hurt myself" → Blocked ✅

### Key Learnings
- Prompt engineering is powerful — a well written system prompt completely 
  changes how a model behaves
- Safety filters are essential in any health-related AI application
- Groq API with Llama 3.1 is a reliable and fast free alternative to OpenAI

### Conclusion
This chatbot demonstrates how LLMs can be used responsibly in healthcare 
contexts with proper prompt design and safety measures in place.